In [14]:
# Here we are performing sentiment analysis(RNN) on data set of 50,000 IMDB reviews (there are results in the dataset but we will not use it).

In [2]:
import pandas as pd

In [41]:
df= pd.read_csv("IMDB Dataset.csv")

In [42]:
df.shape

(50000, 2)

In [45]:
df.drop_duplicates(inplace= True)

In [47]:
df.shape

(49582, 2)

## Text Pre-processing

In [50]:
df["review"]= df["review"].str.lower()

In [52]:
import re # for url removal using regex

In [54]:
def remove_urls(text): 
    text= re.sub(r"https\S+" ,"",text) # (patterrn to remove, what to pLace, i/p text)
    return text
df["review"]= df["review"].apply(remove_urls)

In [56]:
def remove_punc(text):  # removing punctuations
    text= re.sub(r"[^A-Za-z0-9\s]", "", text) # square brackets and ^ means excluding these(\s means space) remove other things with ""(nothing)
    return text 

df["review"]= df["review"].apply(remove_punc)

In [57]:
def html_pattern(text):
    text= re.sub(r"<.*?>","", text) 
    return text 

df["review"]= df["review"].apply(html_pattern)

In [60]:
import nltk # natural language tool-kit 

nltk.download("punkt") # punkt- official module for tockenisation
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\VICTUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\VICTUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VICTUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [62]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [64]:
def remove_stopwords(text): 
    tokens= word_tokenize(text)
    stop_word= stopwords.words("english") # this will filter out all stopwords from english lang. from our tokens

    for word in tokens:
        if word in stop_word: 
            text = text.replace(word,"")

    return text 

df["review"]= df["review"].apply(remove_stopwords)

In [66]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [68]:
# stemming
from nltk.stem import PorterStemmer 

In [70]:
def stemming(text):
    ps= PorterStemmer()
    stemmed_words= []

    tokens= word_tokenize(text) # tokenizing again before stemming
    for token in tokens: 
        stemmed_token= ps.stem(token) # this "stem" will convert each token 
        stemmed_words.append(stemmed_token) # list 

    return " ".join(stemmed_words) # converting to string by combining with empty string 

df["review"]= df["review"].apply(stemming)

In [72]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [74]:
# encoding
from sklearn.preprocessing import LabelEncoder 
le= LabelEncoder()

df["sentiment"]= le.fit_transform(df["sentiment"])

In [76]:
y= df["sentiment"]

In [80]:
# vectorization 
from sklearn.feature_extraction.text import TfidfVectorizer

# there are a lot of words and it will form a huge vector matrix with its defalt parameters so we are limiting it to top n no. of words.

tf= TfidfVectorizer(max_features= 5000) # each feature is a word and these are most important ones. 
X= tf.fit_transform(df["review"])

In [84]:
X

<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 4057154 stored elements in Compressed Sparse Row format>

## Dataset & Dataloaders

In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X,y, random_state=42, test_size=0.2)

In [93]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

In [95]:
X_train= X_train.toarray() # converting matrix into array for making tensor 
X_test= X_test.toarray()

In [103]:
train_set= TensorDataset(
    torch.from_numpy(X_train).float(), # Directly converting into tensor 
    torch.from_numpy(y_train.values).float()
)

test_set= TensorDataset(
    torch.from_numpy(X_test).float(), 
    torch.from_numpy(y_test.values).float()
)

In [105]:
train_loader= DataLoader(train_set, shuffle=True ,batch_size= 64)
test_loader= DataLoader(test_set, shuffle=True ,batch_size= 64)

## Building our RNN

In [118]:
import torch.nn as nn
import torch.optim as optim

In [120]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1): # creating a single layer rnn with 128 hidden layers of some i/p size. 
        super().__init__()

        self.hidden_size= hidden_size
        self.num_layers= num_layers

        # We already have rnn in torch so we just have to define it. 
        self.rnn= nn.RNN(input_size, hidden_size, num_layers, batch_first= True)

        # Fully connected layer 
        self.fc= nn.Linear(hidden_size, 1) # its a many to one architecture 1 = o/p

    def forward(self, x): 
        # we are gong to initialise first hidden state which is optional to do ,but done to see its dimensions 
        h0= torch.zeros(self.num_layers, x.size(0) ,self.hidden_size) # shape(no. of layers, batch size, hidden size) 

        out, _= self.rnn(x, h0) # 1st value= hidden state of all time steps(batch, seq_len, hidden size), 2nd value= final hidden state of last timestep 
        out= self.fc(out[:, -1, :]) # we are taking all batches ,last timestep, and all hidden state ,and o/p will be probabilistic value. 

        return out

In [122]:
input_size= X_train.shape[1]
model= RNN(input_size)
criterion = nn.BCELoss() #binary croo entropy loss used when there are only 2 o/p
optimizer= optim.Adam(model.parameters())

## Training 

In [127]:
epochs= 10 
for epoch in range(epochs): 
    model.train()

    for Xb ,yb in train_loader:
        optimizer.zero_grad()
        
        Xb = Xb.unsqueeze(1) # convertin tfidf vectors(2D) into 3D by adding as singleton direction
        outputs= model(Xb) # raw outcomes in 2D (batch_size, 1)

        outputs= torch.sigmoid(outputs.squeeze()) # converting into 1D (batch_size,), and geting the final probability outcome

        loss = criterion(outputs, yb) # computing loss 
        loss.backward() #Back propagation
        optimizer.step() # weight updation 

    print(f"epoch= {epoch+1}/{epochs} and loss= {loss.item()}")

epoch= 1/10 and loss= 0.262037068605423
epoch= 2/10 and loss= 0.20168505609035492
epoch= 3/10 and loss= 0.33109593391418457
epoch= 4/10 and loss= 0.36199530959129333
epoch= 5/10 and loss= 0.17742429673671722
epoch= 6/10 and loss= 0.48133566975593567
epoch= 7/10 and loss= 0.3029635548591614
epoch= 8/10 and loss= 0.14083802700042725
epoch= 9/10 and loss= 0.31217387318611145
epoch= 10/10 and loss= 0.41993752121925354


In [129]:
# Evaluation 
model.eval()

with torch.no_grad(): 
    correct_vals=0
    tot_vals= 0
    
    for Xb, yb in test_loader:
        Xb= Xb.unsqueeze(1)
        outputs= model(Xb) 
        predicted= (torch.sigmoid(outputs.squeeze())>0.5).float() # if value is grater then 0.5 it will be true=1

        tot_vals+= yb.size(0)
        correct_vals+=(predicted== yb).sum().item()

    print(f"Accuracy= {correct_vals/tot_vals*100}")

Accuracy= 85.5803166280125
